# bce-log-loss-real-fake — faded example 2: Build the real-image target tensor

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `bce-log-loss-real-fake`. Running the beacon reports progress on the `GAN: BCE log loss real/fake` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BCE log loss real/fake` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bce-log-loss-real-fake`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bce-log-loss-real-fake"
DD_SUBTOPIC = "GAN: BCE log loss real/fake"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

BCE needs a target tensor the same shape and dtype as the prediction. For the discriminator's real term, D should output probability 1, so the target is an all-ones tensor matching `d_pred_real`. `t.ones_like` produces exactly that without you hand-coding the shape.

## Faded exercise 2

Implement `disc_loss(d_pred_real, d_pred_fake)`. The fake target and both BCE calls are provided. You must complete the **real target tensor** so that `F.binary_cross_entropy(d_pred_real, real_targets)` penalizes D for not predicting 1 on real images.

**Fill in:** an all-ones tensor matching the shape/dtype of `d_pred_real` (D's target on real images)

In [ ]:
import torch.nn.functional as F

def disc_loss(d_pred_real: t.Tensor, d_pred_fake: t.Tensor) -> t.Tensor:
    real_targets = None  # TODO: an all-ones tensor matching the shape/dtype of d_pred_real
    fake_targets = t.zeros_like(d_pred_fake)
    loss_real = F.binary_cross_entropy(d_pred_real, real_targets)
    loss_fake = F.binary_cross_entropy(d_pred_fake, fake_targets)
    return loss_real + loss_fake

t.manual_seed(0)
d_pred_real = t.rand(8)
d_pred_fake = t.rand(8)
print(round(disc_loss(d_pred_real, d_pred_fake).item(), 6))


def _test():
    import torch.nn.functional as F
    t.manual_seed(2)
    pr = t.rand(12)
    pf = t.rand(12)
    expected = (F.binary_cross_entropy(pr, t.ones_like(pr))
                + F.binary_cross_entropy(pf, t.zeros_like(pf)))
    got = disc_loss(pr, pf)
    assert got.shape == t.Size([]), f'expected scalar, got shape {got.shape}'
    assert t.allclose(got, expected, atol=1e-6), f'{got.item()} vs {expected.item()}'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def disc_loss(d_pred_real: t.Tensor, d_pred_fake: t.Tensor) -> t.Tensor:
    real_targets = t.ones_like(d_pred_real)
    fake_targets = t.zeros_like(d_pred_fake)
    loss_real = F.binary_cross_entropy(d_pred_real, real_targets)
    loss_fake = F.binary_cross_entropy(d_pred_fake, fake_targets)
    return loss_real + loss_fake

t.manual_seed(0)
d_pred_real = t.rand(8)
d_pred_fake = t.rand(8)
print(round(disc_loss(d_pred_real, d_pred_fake).item(), 6))
```
</details>